# Walk-Forward Validation

### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

### DataFrame Import

In [2]:
df = pd.read_pickle('momentum_df.pkl')

df.head()

,Mom,Mkt-RF,SMB,HML,RF,VIX,VIX_lag,RealVol,RealVol_lag,Mkt12m,Mkt12m_lag,MomRecent,MomRecent_lag,Excess_Returns
2000-01-01,0.0186,-0.0474,0.0516,-0.0112,0.0041,24.95,24.64,0.077962,NaN,NaN,NaN,NaN,NaN,0.0145
2000-02-01,0.1802,0.0246,0.2125,-0.0977,0.0043,23.37,24.95,0.054776,0.077962,NaN,NaN,NaN,NaN,0.1759
2000-03-01,-0.0685,0.0521,-0.1741,0.0850,0.0047,24.11,23.37,0.077523,0.054776,NaN,NaN,NaN,NaN,-0.0732
2000-04-01,-0.0860,-0.0639,-0.0600,0.0645,0.0046,26.20,24.11,0.113517,0.077523,NaN,NaN,NaN,NaN,-0.0906
2000-05-01,-0.0899,-0.0439,-0.0608,0.0459,0.0050,23.65,26.20,0.081499,0.113517,NaN,NaN,NaN,NaN,-0.0949


### Expanding-Window

In [3]:
signals = ['VIX_lag', 'Mkt12m_lag']
results = []
exposures = []

for year in range(2010, 2025):
    train = df.loc['2000':str(year - 1)]
    test = df.loc[str(year):str(year)]

    zscores = {}
    for signal in signals:
        train_mean = train[signal].mean()
        train_std = train[signal].std()
        zscores[signal] = [train_mean, train_std]

    train_scores = pd.DataFrame(index=train.index)
    for signal, vals in zscores.items():
        name = signal + "_z"
        train_scores[name] = (train[signal] - vals[0]) / vals[1]
    train_scores['Mkt12m_lag_z'] = -train_scores['Mkt12m_lag_z']
    train_scores['risk_score'] = 0.5 * train_scores['VIX_lag_z'] + 0.5 * train_scores['Mkt12m_lag_z']

    test_scores = pd.DataFrame(index=test.index)
    for signal, vals in zscores.items():
        name = signal + "_z"
        test_scores[name] = (test[signal] - vals[0]) / vals[1]
    test_scores['Mkt12m_lag_z'] = -test_scores['Mkt12m_lag_z']
    test_scores['risk_score'] = 0.5 * test_scores['VIX_lag_z'] + 0.5 * test_scores['Mkt12m_lag_z']

    p_lo = train_scores['risk_score'].quantile(0.10)
    p_hi = train_scores['risk_score'].quantile(0.90)

    exposure = 1 - (test_scores['risk_score'] - p_lo) / (p_hi - p_lo)
    exposure = exposure.clip(0, 1)

    test_excess_returns = pd.Series(
        exposure.values * (test['Mom'] - test['RF']).values,
        index=test.index
    )

    results.append(test_excess_returns)
    exposures.append(pd.Series(exposure.values, index=test.index))  # added

oos = pd.concat(results)
oos_exposure = pd.concat(exposures)

### Metrics

In [4]:
sharpe = (oos.mean() / oos.std()) * np.sqrt(12)
downside_dev = np.sqrt(np.mean(np.minimum(oos, 0)**2))
sortino = (oos.mean() / downside_dev) * np.sqrt(12)
cumulative = (1 + oos).cumprod()
max_dd = ((cumulative - cumulative.cummax()) / cumulative.cummax()).min()
print(f"WF OOS Sharpe: {sharpe:.3f} | Sortino: {sortino:.3f} | Max DD: {max_dd:.3f}")

WF OOS Sharpe: 0.401 | Sortino: 0.636 | Max DD: -0.151


In [5]:
baseline_results = []
for year in range(2010, 2025):
    test = df.loc[str(year):str(year)]
    baseline_excess = test['Mom'] - test['RF']
    baseline_results.append(baseline_excess)

oos_baseline = pd.concat(baseline_results)
sharpe = (oos_baseline.mean() / oos_baseline.std()) * np.sqrt(12)
downside_dev = np.sqrt(np.mean(np.minimum(oos_baseline, 0)**2))
sortino = (oos_baseline.mean() / downside_dev) * np.sqrt(12)
cumulative = (1 + oos_baseline).cumprod()
max_dd = ((cumulative - cumulative.cummax()) / cumulative.cummax()).min()
print(f"WF Baseline Sharpe: {sharpe:.3f} | Sortino: {sortino:.3f} | Max DD: {max_dd:.3f}")

WF Baseline Sharpe: 0.147 | Sortino: 0.199 | Max DD: -0.285


In [6]:
for label, start, end in [('2010-2017', '2010', '2017'), ('2018-2024', '2018', '2024')]:
    sub_dynamic = oos.loc[start:end]
    sub_base = oos_baseline.loc[start:end]
    sharpe_d = (sub_dynamic.mean() / sub_dynamic.std()) * np.sqrt(12)
    sharpe_b = (sub_base.mean() / sub_base.std()) * np.sqrt(12)
    print(f"{label} | Dynamic: {sharpe_d:.3f} | Baseline: {sharpe_b:.3f}")

2010-2017 | Dynamic: 0.408 | Baseline: 0.294
2018-2024 | Dynamic: 0.392 | Baseline: 0.034


### Crash-Conditional Analysis

In [7]:
crash_threshold = df['2000':'2009']['Mom'].quantile(0.10)
crash_mask = oos_baseline[oos_baseline <= crash_threshold].index
print(f"Baseline crash-conditional mean return: {oos_baseline[crash_mask].mean():.4f}")
print(f"Dynamic crash-conditional mean return: {oos[crash_mask].mean():.4f}")
print(f"Mean OOS exposure: {oos_exposure.mean():.4f}")

Baseline crash-conditional mean return: -0.1264
Dynamic crash-conditional mean return: -0.0208
Mean OOS exposure: 0.7402


### Jobson-Korkie Test

In [8]:
sr_dynamic = oos.mean() / oos.std()
sr_baseline = oos_baseline.mean() / oos_baseline.std()

T = len(oos)
rho = oos.corr(oos_baseline)

asymptotic_var = (1 / T) * (2 - (2 * rho * sr_dynamic * sr_baseline) + 0.5 * (sr_dynamic ** 2) + 0.5 * (sr_baseline ** 2) - (rho ** 2) * sr_dynamic * sr_baseline)

z = (sr_dynamic - sr_baseline) / np.sqrt(asymptotic_var)
p_value = stats.norm.sf(z)

print(f"z-statistic: {z:.3f}")
print(f"p-value: {p_value:.4f}")

z-statistic: 0.697
p-value: 0.2430


### Factor Attribution

In [9]:
factors = df.loc[oos.index, ['Mkt-RF', 'SMB', 'HML']]

X = sm.add_constant(factors)
model = sm.OLS(oos, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.127
Model:                            OLS   Adj. R-squared:                  0.112
Method:                 Least Squares   F-statistic:                     8.512
Date:                Tue, 31 Mar 2026   Prob (F-statistic):           2.60e-05
Time:                        11:08:34   Log-Likelihood:                 440.55
No. Observations:                 180   AIC:                            -873.1
Df Residuals:                     176   BIC:                            -860.3
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0027      0.002      1.665      0.0

### Key Findings and Summary

#### Walk-Forward Validation Results (2010–2024 OOS)
- Baseline:
    - Sharpe: 0.147
    - Sortino: 0.199
    - Max DD: -0.285
- Dynamic (continuous scaling):
    - Sharpe: 0.401
    - Sortino: 0.636
    - Max DD: -0.151

- Sub-period Sharpe (2010–2017): Dynamic 0.408 | Baseline 0.294
- Sub-period Sharpe (2018–2024): Dynamic 0.392 | Baseline 0.034

#### Statistical Testing
- Jobson-Korkie 
    - z-statistic: 0.697
    - p-value: 0.2430
    - Sharpe improvement is not statistically significant (n=180 months)

#### FF3 Factor Attribution (OOS Dynamic Excess Returns)
- Alpha: 0.0027 (p-value = 0.098)
- Mkt-RF: -0.043 (p-value = 0.271)
- SMB: -0.057 (p-value = 0.387)
- HML: -0.220 (p-value < 0.001)

- R-squared = 0.127
- Monthly alpha of 0.27% (3.2% annualized) is not statistically significant at the 5% level (p=0.098)
- No significant market or size exposure
- Negative HML loading reflects momentum's structural growth tilt and is mechanical, not a deliberate design choice

#### Limitations
- Jobson-Korkie test is underpowered at 180 monthly observations
- Signal weights (0.50 VIX_lag / 0.50 Mkt12m_lag) selected on economic rationale after logistic regression exclusion of real volatility (p-value = 0.952) and t-test exclusion of lagged momentum (p=0.56).
- Transaction cost model applies costs to overlay weight changes only, not underlying long-short portfolio turnover
- Negative HML loading is mechanical, not a deliberate design choice
- Post-2009 structural break in momentum means OOS regime differs substantially from training regime